
# Amazon Search — Week 7 V4
## Final Evaluation, System Validation & Report Package

This is the final Week 7 integration step.

### Upload only
`amazon_week7_v3_streamlit_demo.zip`

### What V4 does
1. validates the complete V3 package;
2. checks FastAPI + Streamlit source structure;
3. loads the real 10K catalog / FAISS / 29-feature LambdaMART system;
4. runs end-to-end search smoke tests;
5. benchmarks fresh and cached search latency;
6. validates FastAPI `/health` and `/search`;
7. generates the final architecture diagram;
8. generates latency-evolution charts;
9. writes a complete Week 7 technical report in Markdown;
10. packages all final Week 7 evidence into one ZIP.

### Previously validated milestones
- V1.1 functional integration: **PASS**
- V1.1 mean latency: **212.86 ms**
- V1.3 fresh mean: **83.94 ms**
- V1.3 fresh P95: **99.91 ms**
- V1.3 cached mean: **28.84 ms**
- V2 API round-trip mean: **92.57 ms**
- V2 API P95: **119.59 ms**
- V3 Streamlit demo: **PASS**


In [ ]:

# 1. Install dependencies
!pip -q install \
    pandas pyarrow numpy scikit-learn faiss-cpu lightgbm \
    sentence-transformers rank-bm25 transformers accelerate \
    fastapi uvicorn httpx requests streamlit matplotlib

print("✅ Dependencies installed")


In [ ]:

# 2. Upload and extract V3 package
from google.colab import files
from pathlib import Path
import zipfile, shutil, json, os, re, time, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("/content/amazon_week7_v4")
if ROOT.exists():
    shutil.rmtree(ROOT)

ROOT.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

zip_candidates = [x for x in uploaded if x.lower().endswith(".zip")]

v3_zip = None
for name in zip_candidates:
    low = name.lower()
    if "week7" in low and "v3" in low and "streamlit" in low:
        v3_zip = name
        break

if v3_zip is None:
    if len(zip_candidates) == 1:
        v3_zip = zip_candidates[0]
    else:
        raise ValueError(
            "Please upload amazon_week7_v3_streamlit_demo.zip"
        )

zip_path = ROOT / v3_zip
zip_path.write_bytes(uploaded[v3_zip])

SERVICE_ROOT = ROOT / "service"
FINAL_OUT = ROOT / "final_evaluation"
SERVICE_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_OUT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(SERVICE_ROOT)

print("✅ Extracted:", v3_zip)


In [ ]:

# 3. Package structure validation
required_files = [
    "Dockerfile",
    "requirements.txt",
    "README.md",
    "run_demo.py",
    "docker-compose.yml",
    "streamlit_app.py",
    "test_api.py",
    "amazon_search_service/__init__.py",
    "amazon_search_service/main.py",
    "amazon_search_service/search_engine.py",
    "artifacts/catalog.parquet",
    "artifacts/sbert_product_embeddings.npy",
    "artifacts/clip_image_embeddings.npy",
    "artifacts/clip_embedding_map.csv",
    "artifacts/lambdamart_model.txt",
    "artifacts/runtime_config.json",
    "evaluation/week7_v2_summary.json",
    "evaluation/week7_v3_summary.json",
]

structure_rows = []

for rel in required_files:
    exists = (SERVICE_ROOT / rel).exists()
    structure_rows.append({
        "check": rel,
        "status": "PASS" if exists else "FAIL"
    })

structure_df = pd.DataFrame(structure_rows)
display(structure_df)

if (structure_df["status"] == "FAIL").any():
    raise FileNotFoundError(
        "Required V3 package files are missing."
    )

print("✅ Package structure passed")


In [ ]:

# 4. Static code validation
import py_compile

python_files = [
    "streamlit_app.py",
    "run_demo.py",
    "test_api.py",
    "amazon_search_service/main.py",
    "amazon_search_service/search_engine.py",
]

static_rows = []

for rel in python_files:
    try:
        py_compile.compile(
            str(SERVICE_ROOT / rel),
            doraise=True
        )
        status = "PASS"
        detail = "Syntax valid"
    except Exception as e:
        status = "FAIL"
        detail = str(e)

    static_rows.append({
        "check": f"syntax::{rel}",
        "status": status,
        "detail": detail,
    })

streamlit_source = (
    SERVICE_ROOT / "streamlit_app.py"
).read_text(encoding="utf-8")

api_source = (
    SERVICE_ROOT / "amazon_search_service/main.py"
).read_text(encoding="utf-8")

ui_terms = [
    "st.text_input",
    "st.button",
    "ranking_score",
    "retrieval_score",
    "latency_ms",
    "st.image",
    "top_k",
    "candidate_k",
]

for term in ui_terms:
    static_rows.append({
        "check": f"ui::{term}",
        "status": "PASS" if term in streamlit_source else "FAIL",
        "detail": "Required Streamlit element"
    })

api_terms = [
    '@app.get("/health")',
    '"/search"',
    "SearchEngine",
]

for term in api_terms:
    static_rows.append({
        "check": f"api::{term}",
        "status": "PASS" if term in api_source else "FAIL",
        "detail": "Required FastAPI element"
    })

static_df = pd.DataFrame(static_rows)
display(static_df)

if (static_df["status"] == "FAIL").any():
    raise AssertionError("Static validation failed")

print("✅ Static source validation passed")


In [ ]:

# 5. Load prior validated V2/V3 summaries
with open(
    SERVICE_ROOT / "evaluation/week7_v2_summary.json"
) as f:
    v2_summary = json.load(f)

with open(
    SERVICE_ROOT / "evaluation/week7_v3_summary.json"
) as f:
    v3_summary = json.load(f)

v2_api_mean = float(
    v2_summary["api_roundtrip_mean_ms"]
)
v2_api_p95 = float(
    v2_summary["api_roundtrip_p95_ms"]
)

print("V2 API mean :", round(v2_api_mean, 3), "ms")
print("V2 API P95 :", round(v2_api_p95, 3), "ms")
print("V3 demo pass:", all(v3_summary["demo_features"].values()))


In [ ]:

# 6. Load the actual packaged SearchEngine
import sys
sys.path.insert(0, str(SERVICE_ROOT))

from amazon_search_service.search_engine import SearchEngine

engine = SearchEngine(
    artifact_dir=SERVICE_ROOT / "artifacts"
)

health = engine.health()
print(json.dumps(health, indent=2))

runtime_checks = {
    "catalog_10000":
        health.get("catalog_products") == 10000,
    "faiss_10000":
        health.get("faiss_products") == 10000,
    "ranker_29_features":
        health.get("ranker_features") == 29,
    "engine_status_ok":
        health.get("status") == "ok",
}

for k,v in runtime_checks.items():
    print(("✅" if v else "❌"), k)

if not all(runtime_checks.values()):
    raise AssertionError("Runtime health validation failed")

print("✅ Runtime health passed")


In [ ]:

# 7. End-to-end search quality smoke tests
quality_queries = [
    "best non drying lip stain",
    "came with no tanning mitt",
    "perfect scent for a romantic evening",
]

quality_rows = []
quality_latency_rows = []

for q in quality_queries:
    engine.clear_caches()

    result = engine.search(
        q,
        top_k=10,
        candidate_k=100
    )

    print("\n" + "=" * 100)
    print("QUERY:", q)
    print(
        "Latency:",
        round(result["latency_ms"]["total"], 2),
        "ms"
    )

    result_df = pd.DataFrame(result["results"])
    display(
        result_df[
            [
                "rank",
                "title",
                "brand",
                "ranking_score",
                "retrieval_score",
            ]
        ]
    )

    temp = result_df.copy()
    temp.insert(0, "query", q)
    quality_rows.append(temp)

    quality_latency_rows.append({
        "query": q,
        **result["latency_ms"]
    })

quality_results = pd.concat(
    quality_rows,
    ignore_index=True
)
quality_latency = pd.DataFrame(
    quality_latency_rows
)

quality_results.to_csv(
    FINAL_OUT / "week7_v4_quality_smoke_results.csv",
    index=False
)
quality_latency.to_csv(
    FINAL_OUT / "week7_v4_quality_smoke_latency.csv",
    index=False
)

print("✅ Search quality smoke tests completed")


In [ ]:

# 8. Final fresh-query latency benchmark
# Use known queries already present in prior V2 benchmark when available.
benchmark_path = (
    SERVICE_ROOT
    / "evaluation"
    / "week7_v2_api_latency_benchmark.csv"
)

if benchmark_path.exists():
    prior_bench = pd.read_csv(benchmark_path)
    benchmark_queries = (
        prior_bench["query"]
        .dropna()
        .astype(str)
        .tolist()
    )
else:
    benchmark_queries = quality_queries.copy()

benchmark_queries = list(dict.fromkeys(benchmark_queries))[:10]

# Warm model kernels once.
_ = engine.search(
    benchmark_queries[0],
    top_k=10,
    candidate_k=100
)

fresh_rows = []

for q in benchmark_queries:
    engine.clear_caches()

    result = engine.search(
        q,
        top_k=10,
        candidate_k=100
    )

    fresh_rows.append({
        "query": q,
        **result["latency_ms"]
    })

fresh_df = pd.DataFrame(fresh_rows)

fresh_mean = float(
    fresh_df["total"].mean()
)
fresh_median = float(
    fresh_df["total"].median()
)
fresh_p95 = float(
    fresh_df["total"].quantile(0.95)
)
fresh_max = float(
    fresh_df["total"].max()
)

display(fresh_df)

print(f"Fresh mean   : {fresh_mean:.2f} ms")
print(f"Fresh median : {fresh_median:.2f} ms")
print(f"Fresh P95    : {fresh_p95:.2f} ms")
print(f"Fresh max    : {fresh_max:.2f} ms")
print(
    "Fresh mean <200ms:",
    "✅ PASS"
    if fresh_mean < 200
    else "⚠️ ABOVE TARGET"
)

fresh_df.to_csv(
    FINAL_OUT / "week7_v4_fresh_latency_benchmark.csv",
    index=False
)


In [ ]:

# 9. Final cached-query latency benchmark
engine.clear_caches()

# Populate caches.
for q in benchmark_queries:
    _ = engine.search(
        q,
        top_k=10,
        candidate_k=100
    )

cached_rows = []

for q in benchmark_queries:
    result = engine.search(
        q,
        top_k=10,
        candidate_k=100
    )

    cached_rows.append({
        "query": q,
        **result["latency_ms"]
    })

cached_df = pd.DataFrame(cached_rows)

cached_mean = float(
    cached_df["total"].mean()
)
cached_median = float(
    cached_df["total"].median()
)
cached_p95 = float(
    cached_df["total"].quantile(0.95)
)

display(cached_df)

print(f"Cached mean   : {cached_mean:.2f} ms")
print(f"Cached median : {cached_median:.2f} ms")
print(f"Cached P95    : {cached_p95:.2f} ms")

cached_df.to_csv(
    FINAL_OUT / "week7_v4_cached_latency_benchmark.csv",
    index=False
)


In [ ]:

# 10. Final FastAPI functional test
from fastapi.testclient import TestClient
from amazon_search_service.main import app

api_checks = {}

with TestClient(app) as client:
    health_response = client.get("/health")
    api_checks["GET_health_200"] = (
        health_response.status_code == 200
    )

    search_response = client.post(
        "/search",
        json={
            "query": "best non drying lip stain",
            "top_k": 5,
            "candidate_k": 100,
        }
    )

    api_checks["POST_search_200"] = (
        search_response.status_code == 200
    )

    if search_response.status_code == 200:
        body = search_response.json()

        api_checks["five_results"] = (
            len(body["results"]) == 5
        )
        api_checks["rank_starts_1"] = (
            body["results"][0]["rank"] == 1
        )
        api_checks["ranking_score_present"] = (
            "ranking_score"
            in body["results"][0]
        )
        api_checks["retrieval_score_present"] = (
            "retrieval_score"
            in body["results"][0]
        )
        api_checks["latency_present"] = (
            "latency_ms" in body
        )
    else:
        for key in [
            "five_results",
            "rank_starts_1",
            "ranking_score_present",
            "retrieval_score_present",
            "latency_present",
        ]:
            api_checks[key] = False

    bad_response = client.post(
        "/search",
        json={
            "query": "lip stain",
            "top_k": 10,
            "candidate_k": 5,
        }
    )

    api_checks["invalid_request_422"] = (
        bad_response.status_code == 422
    )

for k,v in api_checks.items():
    print(("✅" if v else "❌"), k)

if not all(api_checks.values()):
    raise AssertionError(
        "Final API validation failed"
    )

print("\n✅ Final FastAPI validation passed")


In [ ]:

# 11. Aggregate final validation matrix
validation_rows = []

def add_group(group, mapping):
    for k,v in mapping.items():
        validation_rows.append({
            "group": group,
            "check": k,
            "status": "PASS" if bool(v) else "FAIL"
        })

add_group(
    "runtime",
    runtime_checks
)

add_group(
    "api",
    api_checks
)

add_group(
    "v2_validation",
    v2_summary.get(
        "validation",
        {}
    )
)

add_group(
    "v3_demo_features",
    v3_summary.get(
        "demo_features",
        {}
    )
)

final_validation = pd.DataFrame(
    validation_rows
)

display(final_validation)

final_validation.to_csv(
    FINAL_OUT / "week7_v4_final_validation_checks.csv",
    index=False
)

failed_checks = (
    final_validation["status"] == "FAIL"
).sum()

print(
    f"\nFinal validation: "
    f"{len(final_validation)-failed_checks}/"
    f"{len(final_validation)} PASS"
)

if failed_checks:
    raise AssertionError(
        f"{failed_checks} final checks failed"
    )


In [ ]:

# 12. Architecture diagram
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")

boxes = [
    (0.5, 5.2, 2.0, 1.0, "User Query"),
    (3.0, 5.2, 2.0, 1.0, "Streamlit UI"),
    (5.5, 5.2, 2.0, 1.0, "FastAPI\\n/search"),
    (8.0, 5.2, 2.0, 1.0, "SBERT Query\\nEmbedding"),
    (10.5, 5.2, 2.0, 1.0, "FAISS\\nTop-100"),
    (10.5, 2.8, 2.0, 1.0, "29 Runtime\\nFeatures"),
    (8.0, 2.8, 2.0, 1.0, "LambdaMART\\nRe-ranking"),
    (5.5, 2.8, 2.0, 1.0, "Top-K Ranked\\nProducts"),
]

for x,y,w,h,label in boxes:
    patch = FancyBboxPatch(
        (x,y),
        w,h,
        boxstyle="round,pad=0.05",
        linewidth=1.5,
        fill=False
    )
    ax.add_patch(patch)
    ax.text(
        x+w/2,
        y+h/2,
        label,
        ha="center",
        va="center",
        fontsize=11
    )

arrows = [
    ((2.5,5.7),(3.0,5.7)),
    ((5.0,5.7),(5.5,5.7)),
    ((7.5,5.7),(8.0,5.7)),
    ((10.0,5.7),(10.5,5.7)),
    ((11.5,5.2),(11.5,3.8)),
    ((10.5,3.3),(10.0,3.3)),
    ((8.0,3.3),(7.5,3.3)),
]

for start,end in arrows:
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="->",
            mutation_scale=15,
            linewidth=1.5
        )
    )

ax.text(
    7,
    7.1,
    "Amazon Search — Week 7 Final End-to-End Architecture",
    ha="center",
    fontsize=16,
    fontweight="bold"
)

ax.text(
    7,
    1.2,
    "10K Product Catalog · Week 4 SBERT/CLIP · Week 5 LambdaMART · Top-100 Retrieval",
    ha="center",
    fontsize=10
)

plt.tight_layout()
plt.savefig(
    FINAL_OUT / "week7_v4_architecture.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()


In [ ]:

# 13. Latency evolution plot
import matplotlib.pyplot as plt

latency_history = pd.DataFrame({
    "version": [
        "V1.1 Functional",
        "V1.3 Optimized",
        "V2 API Round-trip",
        "V4 Fresh Recheck",
    ],
    "mean_latency_ms": [
        212.86,
        83.94,
        v2_api_mean,
        fresh_mean,
    ]
})

display(latency_history)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    latency_history["version"],
    latency_history["mean_latency_ms"]
)

ax.axhline(
    200,
    linestyle="--",
    linewidth=1.5,
    label="Week 7 target = 200 ms"
)

ax.set_ylabel("Mean latency (ms)")
ax.set_title(
    "Week 7 — End-to-End Latency Evolution"
)
ax.tick_params(
    axis="x",
    rotation=15
)
ax.legend()

plt.tight_layout()
plt.savefig(
    FINAL_OUT / "week7_v4_latency_evolution.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

latency_history.to_csv(
    FINAL_OUT / "week7_v4_latency_evolution.csv",
    index=False
)


In [ ]:

# 14. Stage latency plot
stage_means = pd.DataFrame({
    "stage": [
        "SBERT embedding",
        "FAISS retrieval",
        "Feature generation",
        "LambdaMART ranking",
    ],
    "mean_ms": [
        fresh_df["embedding"].mean(),
        fresh_df["retrieval"].mean(),
        fresh_df["features"].mean(),
        fresh_df["ranking"].mean(),
    ]
})

display(stage_means)

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    stage_means["stage"],
    stage_means["mean_ms"]
)

ax.set_ylabel("Mean latency (ms)")
ax.set_title(
    "Week 7 V4 — Fresh Query Stage Latency"
)
ax.tick_params(
    axis="x",
    rotation=20
)

plt.tight_layout()
plt.savefig(
    FINAL_OUT / "week7_v4_stage_latency.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

stage_means.to_csv(
    FINAL_OUT / "week7_v4_stage_latency.csv",
    index=False
)


In [ ]:

# 15. Final metrics table
v11_latency = 212.86
v13_fresh_mean = 83.94
v13_fresh_p95 = 99.91
v13_cached_mean = 28.84

optimization_pct = (
    (v11_latency - v13_fresh_mean)
    / v11_latency
    * 100
)

final_metrics = pd.DataFrame([
    {
        "metric": "Catalog products",
        "value": 10000,
        "target": "10K",
        "status": "PASS"
    },
    {
        "metric": "Retrieval candidate count",
        "value": 100,
        "target": "Top-100",
        "status": "PASS"
    },
    {
        "metric": "LambdaMART runtime features",
        "value": 29,
        "target": "Exact Week5 schema",
        "status": "PASS"
    },
    {
        "metric": "V1.1 mean latency (ms)",
        "value": v11_latency,
        "target": "<200",
        "status": "NOT MET"
    },
    {
        "metric": "V1.3 fresh mean latency (ms)",
        "value": v13_fresh_mean,
        "target": "<200",
        "status": "PASS"
    },
    {
        "metric": "V1.3 fresh P95 latency (ms)",
        "value": v13_fresh_p95,
        "target": "<200",
        "status": "PASS"
    },
    {
        "metric": "V1.3 cached mean latency (ms)",
        "value": v13_cached_mean,
        "target": "<200",
        "status": "PASS"
    },
    {
        "metric": "V2 API round-trip mean (ms)",
        "value": v2_api_mean,
        "target": "<200",
        "status": "PASS"
    },
    {
        "metric": "V2 API round-trip P95 (ms)",
        "value": v2_api_p95,
        "target": "<200",
        "status": "PASS"
    },
    {
        "metric": "V4 fresh recheck mean (ms)",
        "value": fresh_mean,
        "target": "<200",
        "status": (
            "PASS"
            if fresh_mean < 200
            else "NOT MET"
        )
    },
    {
        "metric": "Latency reduction V1.1 → V1.3 (%)",
        "value": optimization_pct,
        "target": "Improve runtime",
        "status": "PASS"
    },
])

display(final_metrics)

final_metrics.to_csv(
    FINAL_OUT / "week7_v4_final_metrics.csv",
    index=False
)


In [ ]:

# 16. Create complete Week 7 technical report (Markdown)
passed = int(
    (final_validation["status"] == "PASS").sum()
)
total = len(final_validation)

top_quality_examples = []

for q in quality_queries:
    qdf = quality_results[
        quality_results["query"] == q
    ].head(3)

    titles = qdf["title"].astype(str).tolist()
    top_quality_examples.append(
        f"- **{q}**: "
        + " | ".join(titles)
    )

report = f"""# Amazon Search / Learning-to-Rank Project — Week 7 Final Report

## 1. Executive Summary

Week 7 focused on converting the previously developed retrieval and ranking experiments into a complete end-to-end search system.

The final system integrates:

- a 10,000-product Amazon catalog;
- SBERT query embeddings;
- FAISS dense candidate retrieval;
- Top-100 candidate generation;
- 29 ranking features covering lexical, semantic, multimodal, metadata, and interaction signals;
- the saved Week 5 LambdaMART model;
- a FastAPI search service;
- a Streamlit interactive demo;
- latency profiling, caching, validation, and deployment files.

The final Week 7 system satisfies the main roadmap requirement of **sub-200 ms retrieval + re-ranking latency**.

## 2. Final Architecture

User Query  
→ Streamlit Interface  
→ FastAPI `/search`  
→ SBERT Query Embedding  
→ FAISS Top-100 Retrieval  
→ 29 Runtime Ranking Features  
→ LambdaMART Re-ranking  
→ Top-K Ranked Products

See `week7_v4_architecture.png`.

## 3. V1 — Initial End-to-End Integration

The first integration successfully connected query embedding, FAISS retrieval, feature generation, and ranking. However, the initial implementation exposed two integration problems:

1. a Week 5 judged query-product table was initially treated as the retrieval catalog instead of the full Week 1 10K catalog;
2. the saved LambdaMART model required the exact Week 5 feature schema rather than a simplified temporary feature vector.

The corrected V1.1 pipeline therefore used:

- `cleaned_products_10k.parquet` as the full catalog;
- Week 4 SBERT embeddings aligned to that catalog;
- Week 4 CLIP image embeddings;
- the exact Week 5 runtime feature ordering;
- the saved Week 5 LambdaMART model.

All model/catalog/schema alignment checks passed.

## 4. Latency Optimization

The corrected V1.1 mean latency was approximately **{v11_latency:.2f} ms**, slightly above the 200 ms roadmap target.

V1.3 optimized runtime without retraining or changing LambdaMART.

Optimizations included:

- loading only the CLIP text tower at query time;
- precomputing normalized catalog text and token sets;
- precomputing metadata arrays;
- vectorizing CLIP candidate similarity;
- caching SBERT query embeddings;
- caching CLIP query embeddings;
- reducing repeated pandas/string processing.

The resulting V1.3 performance was:

| Metric | Result |
|---|---:|
| Fresh-query mean | {v13_fresh_mean:.2f} ms |
| Fresh-query P95 | {v13_fresh_p95:.2f} ms |
| Cached-query mean | {v13_cached_mean:.2f} ms |
| V1.1 → V1.3 latency reduction | {optimization_pct:.1f}% |

This moved the system comfortably below the 200 ms target.

## 5. V2 — FastAPI Service

The optimized pipeline was packaged as a FastAPI service.

Endpoints:

- `GET /`
- `GET /health`
- `POST /search`

A `/search` request accepts a natural-language query, `top_k`, and `candidate_k`, and returns ranked product results with ranking score, retrieval score, and latency breakdown.

Validated V2 API performance:

| Metric | Result |
|---|---:|
| Internal pipeline mean | {float(v2_summary["pipeline_mean_ms"]):.2f} ms |
| API round-trip mean | {v2_api_mean:.2f} ms |
| API round-trip P95 | {v2_api_p95:.2f} ms |
| Mean latency target | PASS |

API validation also confirmed:

- 10K catalog loaded;
- FAISS contains 10K products;
- LambdaMART uses 29 features;
- `/health` returns HTTP 200;
- `/search` returns HTTP 200;
- invalid candidate counts return HTTP 422.

## 6. V3 — Streamlit Demo

A Streamlit interface was added on top of the production search pipeline.

The demo includes:

- natural-language query input;
- Top-K control;
- candidate-count control;
- product rank and title;
- brand and category;
- price, rating, and review count;
- image display when an image URL is available;
- LambdaMART ranking score;
- SBERT retrieval score;
- latency breakdown;
- explicit `<200 ms` performance status.

The demo supports both direct `SearchEngine` mode and API-backed mode.

## 7. V4 — Final Validation

Final validation status: **{passed}/{total} checks passed**.

Current V4 fresh-query recheck:

| Metric | Result |
|---|---:|
| Mean | {fresh_mean:.2f} ms |
| Median | {fresh_median:.2f} ms |
| P95 | {fresh_p95:.2f} ms |
| Maximum | {fresh_max:.2f} ms |
| Mean < 200 ms | {"PASS" if fresh_mean < 200 else "ABOVE TARGET"} |

The system therefore remains functionally integrated after packaging and deployment-oriented refactoring.

## 8. Example Search Quality

Representative Top-3 results from the final pipeline:

{chr(10).join(top_quality_examples)}

These known-query smoke tests are intended as qualitative integration checks. Full ranking-quality claims remain tied to the offline judged-query evaluation from earlier weeks.

## 9. Engineering Deliverables

The final Week 7 package includes:

- modular `SearchEngine`;
- FastAPI application;
- Streamlit application;
- runtime artifacts;
- `requirements.txt`;
- Dockerfile;
- Docker Compose configuration;
- API tests;
- latency benchmarks;
- validation matrices;
- architecture diagram;
- latency-evolution visualization;
- final Week 7 report.

## 10. Key Result

Week 7 successfully transformed the earlier retrieval and Learning-to-Rank work into an operational end-to-end search application.

The strongest engineering result is the reduction of fresh-query latency from approximately **{v11_latency:.2f} ms** to **{v13_fresh_mean:.2f} ms**, a **{optimization_pct:.1f}% reduction**, while retaining the full 10K catalog, Top-100 retrieval, multimodal feature generation, and the original saved LambdaMART ranker.

The FastAPI service also maintains a mean round-trip latency of approximately **{v2_api_mean:.2f} ms**, comfortably within the Week 7 sub-200 ms objective.

## 11. Next Steps

The next development phase can focus on deployment and search-quality robustness rather than basic integration:

- larger or production-scale product catalog;
- persistent FAISS index loading;
- stronger image coverage;
- batch query processing;
- service-level observability;
- cache metrics and eviction policies;
- asynchronous request handling;
- model/version registry;
- online A/B testing;
- click and conversion feedback;
- retrieval and re-ranking monitoring.

Week 7 is complete from an end-to-end system integration perspective.
"""

report_path = FINAL_OUT / "Amazon_Week7_Final_Technical_Report.md"
report_path.write_text(
    report,
    encoding="utf-8"
)

print(report[:5000])
print("\n✅ Full report saved:", report_path)


In [ ]:

# 17. Final summary JSON
from datetime import datetime

final_summary = {
    "week": 7,
    "status": "COMPLETE",
    "catalog_products": 10000,
    "retrieval_candidates": 100,
    "ranker": "LambdaMART",
    "ranker_features": 29,
    "v1_1_mean_latency_ms": 212.86,
    "v1_3_fresh_mean_latency_ms": 83.94,
    "v1_3_fresh_p95_latency_ms": 99.91,
    "v1_3_cached_mean_latency_ms": 28.84,
    "v2_api_roundtrip_mean_ms": v2_api_mean,
    "v2_api_roundtrip_p95_ms": v2_api_p95,
    "v4_fresh_mean_latency_ms": fresh_mean,
    "v4_fresh_p95_latency_ms": fresh_p95,
    "latency_reduction_v1_1_to_v1_3_pct": optimization_pct,
    "roadmap_latency_target_ms": 200,
    "roadmap_latency_target_pass": True,
    "final_validation_passed": passed,
    "final_validation_total": total,
    "fastapi": True,
    "streamlit": True,
    "docker": True,
    "docker_compose": True,
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
}

with open(
    FINAL_OUT / "week7_v4_final_summary.json",
    "w"
) as f:
    json.dump(
        final_summary,
        f,
        indent=2
    )

print(json.dumps(final_summary, indent=2))


In [ ]:

# 18. Package final Week 7 evidence
# Copy the final report/evaluation folder into a clean deliverable.
DELIVERABLE = ROOT / "amazon_week7_v4_final_package"

if DELIVERABLE.exists():
    shutil.rmtree(DELIVERABLE)

DELIVERABLE.mkdir(parents=True)

# Final evaluation evidence
shutil.copytree(
    FINAL_OUT,
    DELIVERABLE / "final_evaluation"
)

# Deployment/application source
for rel in [
    "amazon_search_service",
    "artifacts",
]:
    shutil.copytree(
        SERVICE_ROOT / rel,
        DELIVERABLE / rel
    )

for rel in [
    "streamlit_app.py",
    "run_demo.py",
    "test_api.py",
    "requirements.txt",
    "Dockerfile",
    "docker-compose.yml",
    "README.md",
]:
    shutil.copy2(
        SERVICE_ROOT / rel,
        DELIVERABLE / rel
    )

zip_base = "/content/amazon_week7_v4_final_package"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    DELIVERABLE
)

print("✅ WEEK 7 V4 COMPLETE")
print("✅ Final package:", zip_path)

print("\nFinal evaluation files:")
for p in sorted(
    (DELIVERABLE / "final_evaluation").iterdir()
):
    print(" -", p.name)

files.download(zip_path)



# Week 7 Complete

The V4 output package contains the final application plus evaluation evidence.

Final deliverable:

`amazon_week7_v4_final_package.zip`

After this run, the Week 7 technical story is complete:

**Full 10K catalog → SBERT → FAISS Top-100 → 29 features → LambdaMART → FastAPI → Streamlit, with sub-200 ms latency.**
